# Лабораторная работа № 3 — YOLO11s (вариант 4)
**Яшков П. А.**  
Задача: обнаружение котов на изображениях Cats v3.

Этот блокнот представляет собой очищенную, воспроизводимую версию эксперимента. Первичный блокнот с фактическим журналом выполнения сохранён отдельно как `LR3_Yashkov_original.ipynb`.  
**Важно:** исходный датасет и файл `best.pt` не входят в сдаваемый ZIP: они должны находиться на вашем Google Drive. Указанные цифры теста взяты из реальных файлов CSV/JSON, а не генерируются в этом блокноте.

## 1. Подключение среды и Drive

In [ ]:
%pip install -q ultralytics pyyaml
import json, shutil, torch, yaml, pandas as pd
from pathlib import Path
from google.colab import drive
from ultralytics import YOLO
from IPython.display import display, Image as IPImage

# Подключаем Диск лишь если он ещё не доступен. Не принуждаем remount.
MOUNT = Path('/content/drive')
if not (MOUNT/'MyDrive').is_dir():
    try:
        drive.mount(str(MOUNT))
    except ValueError:
        MOUNT = Path('/content/gdrive_yashkov')
        if not (MOUNT/'MyDrive').is_dir():
            drive.mount(str(MOUNT))
ROOT = MOUNT/'MyDrive'/'OIIS_LR3_Yashkov'
DATA_SRC = ROOT/'dataset'/'Cats.v3-augmented-v1-accurate.yolov11'
RESULTS = ROOT/'results'
MODELS = ROOT/'models'
for p in (RESULTS, MODELS): p.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Проект:', ROOT, '| GPU:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU')

## 2. Подготовка датасета Cats v3

In [ ]:
LOCAL = Path('/content/cats_v3')
assert DATA_SRC.exists(), f'Сначала загрузите датасет в {DATA_SRC}'
shutil.copytree(DATA_SRC, LOCAL, dirs_exist_ok=True)
yaml_candidates = list(LOCAL.rglob('data.yaml'))
assert yaml_candidates, 'В исходном датасете не найден data.yaml'
DATA_ROOT = yaml_candidates[0].parent
with open(yaml_candidates[0], encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
assert cfg['nc'] == 1, f'Ожидался один класс: {cfg}'
assert 'cat' in str(cfg['names']).lower()
folders = {}
for split, variants in [('train',['train']),('val',['valid','val']),('test',['test'])]:
    folders[split] = next((DATA_ROOT/v/'images' for v in variants if (DATA_ROOT/v/'images').is_dir()),None)
    assert folders[split] is not None, f'Папка {split} отсутствует'
    cfg[split] = str(folders[split])
cfg['path'] = str(DATA_ROOT)
DATA_YAML = LOCAL/'data_colab.yaml'
with open(DATA_YAML,'w',encoding='utf-8') as f:yaml.safe_dump(cfg,f,allow_unicode=True)
for k,p in folders.items():print(k,len(list(p.glob('*.jpg')))+len(list(p.glob('*.png')))+len(list(p.glob('*.jpeg'))))
print('Конфигурация готова:', DATA_YAML)

## 3. Загрузка предобученной модели

In [ ]:
PRETRAINED = MODELS/'yolo11s_pretrained.pt'
if not PRETRAINED.exists():
    YOLO('yolo11s.pt')
    shutil.copy2('yolo11s.pt', PRETRAINED)
model = YOLO(str(PRETRAINED))
print('Предобученных классов:',len(model.names),'cat:',[(k,v) for k,v in model.names.items() if v == 'cat'])

## 4. Обучение (выполнять только при необходимости)
Обучение требует GPU; если файл `best.pt` уже существует, повторять этот этап не требуется.

In [ ]:
RUN = RESULTS/'training'/'yolo11s_cats_v3'
BEST = RUN/'weights'/'best.pt'
if BEST.exists():
    print('Обученная модель уже найдена:', BEST)
else:
    assert DEVICE == 0, 'Для обучения подключите GPU (например, Tesla T4)'
    training = YOLO(str(PRETRAINED))
    training.train(data=str(DATA_YAML),epochs=30,imgsz=640,batch=16,device=0,workers=0,
       optimizer='auto',patience=0,save=True,save_period=5,plots=True,
       project=str(RESULTS/'training'),name='yolo11s_cats_v3',exist_ok=True,
       seed=42,amp=True)
    print('Обучение завершено:', BEST)

## 5. Оценивание на независимой тестовой выборке

In [ ]:
assert BEST.exists(), 'Отсутствуют лучшие веса best.pt'
model = YOLO(str(BEST))
m = model.val(data=str(DATA_YAML),split='test',imgsz=640,batch=16,device=DEVICE,
    workers=0,plots=True,project=str(RESULTS),name='test_evaluation',exist_ok=True)
metrics={'precision':float(m.box.mp),'recall':float(m.box.mr),'mAP50':float(m.box.map50),
         'mAP75':float(m.box.map75),'mAP50-95':float(m.box.map)}
(RESULTS/'test_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
print(metrics)

## 6. Детекция на 5 новых изображениях

In [ ]:
IMAGES = ROOT/'internet_images'
images = sorted(p for p in IMAGES.iterdir() if p.suffix.lower() in ('.png','.jpg','.jpeg'))
assert images, f'Сначала поместите фотографии в {IMAGES}'
preds=model.predict(source=[str(p) for p in images],imgsz=640,conf=0.25,
    device=DEVICE,save=True,save_txt=True,save_conf=True,
    project=str(RESULTS),name='internet_predictions',exist_ok=True)
rows=[]
for r in preds:
    confidences = [float(v) for v in r.boxes.conf]
    rows.append({'Изображение':Path(r.path).name,'Обнаружено котов':len(confidences),
        'Макс. уверенность, %':round(max(confidences)*100,2) if confidences else 0})
df=pd.DataFrame(rows)
df.to_csv(RESULTS/'internet_results.csv',index=False,encoding='utf-8-sig')
display(df)

## 7. Графики и итоговая проверка

In [ ]:
for f in [RUN/'results.png',RESULTS/'test_evaluation'/'BoxPR_curve.png',
          RESULTS/'test_evaluation'/'confusion_matrix.png']:
    if f.is_file():
        print(f.name);display(IPImage(filename=str(f),width=850))
for file in [BEST,RESULTS/'test_metrics.json',RESULTS/'internet_results.csv']:
    print(file.name,'OK' if file.exists() else 'НЕ НАЙДЕН')